# Gemma 3 1B — SRD-4 vs Google QAT Q4_0 Comparison

Packs `google/gemma-3-1b-it` (BF16) into a signed SRD-4 `.axm` container,
extracts to GGUF Q4_K_M, then benchmarks it head-to-head against Google's
official QAT GGUF (`google/gemma-3-1b-it-qat-q4_0-gguf`).

**Why this comparison matters**

| Method | bpw | Training change? | Key property |
|---|---|---|---|
| SRD Q4_K_M (this notebook) | ~4.85 | ✗ Post-training only | Stochastic residual dithering + HMAC governance |
| Google QAT Q4_0 | ~4.0 | ✓ Fake-quant during training | Weights trained to tolerate quantization error |

QAT has fewer bits (4.0 vs 4.85 bpw) but has training-time awareness of quantization.
SRD has more bits and cryptographic provenance — no re-training required.
The PPL delta tells us which tradeoff wins at this model scale.

**What this notebook produces**

| Output | Size (est.) | Use |
|---|---|---|
| `gemma3_1b_srd4.axm` | ~0.9 GB | Signed SRD-4 container (~4.5 bpw) |
| `gemma3_1b_srd4_q4km.gguf` | ~670 MB | llama.cpp / PocketPal / llama-server |
| `gemma3_1b_qat_q4_0.gguf` | ~660 MB | Google QAT GGUF (downloaded from HF) |
| `gemma3_1b_met_sidecar.json` | ~10 KB | MET hydration budget reference |
| PPL comparison table | — | SRD vs QAT on WikiText-2 |

**Architecture (Gemma 3 1B)**

| Field | Value |
|---|---|
| Parameters | ~1.0 B |
| vocab_size | 262,144 |
| hidden_size | 1,152 |
| num_layers | 18 |
| num_attention_heads | 4 |
| num_kv_heads | 1 (MQA — single KV head) |
| intermediate_size | 6,912 |
| MLP style | GeGLU |

> **Note:** Gemma 3 1B uses MQA (1 KV head for 4 query heads) — extreme GQA.
> This is the same bottleneck pattern that gave DeepSeek-R1-1.5B 49% faster TG
> than Qwen3-1.7B on OpenCL. Expect faster mobile TG than the architecture size suggests.

**Baseline PPL (from prior benchmarks)**

| Model | Quant | PPL |
|---|---|---|
| Gemma 3 1B Q4_K_M (standard) | 4.85 bpw | 28.90 |
| TinyLlama 1.1B Q4_K_M | 4.85 bpw | 19.39 |
| Qwen3 1.7B Q4_K_M | 4.85 bpw | 21.19 |

> **Patent pending — Orivael Inc.** The SRD container format and signing protocol are the subject of pending patent claims.

**Runtime:** T4 (15 GB VRAM) recommended. A100 faster (~3× pack speed). CPU-only: Cell 3 will be slow (~1.5 h).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — GPU check · clone axiom · install deps · persist AXIOM_MASTER_KEY
#           clone + cmake-build llama.cpp (GGML_CUDA=ON)  (~10-15 min)
# ══════════════════════════════════════════════════════════════════════════════
import os, secrets, subprocess, sys, time
from pathlib import Path

AXIOM_DIR  = Path('/content/axiom')
OUTPUT_DIR = Path('/content/axiom_output')
LLAMACPP   = Path('/content/llama.cpp')
BRANCH     = 'claude/srd-prototype-benchmark-JRtv1'
REPO_URL   = 'https://github.com/orivael-dev/axiom.git'
KEY_FILE   = OUTPUT_DIR / 'axiom_master.key'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── GPU info ──────────────────────────────────────────────────────────────────
import torch
p = torch.cuda.get_device_properties(0)
vram_gb   = p.total_memory / 1024**3
cuda_arch = p.major * 10 + p.minor
print(f'GPU : {p.name}  {vram_gb:.1f} GB VRAM  SM {p.major}.{p.minor}  arch={cuda_arch}')
if vram_gb < 4:
    print('  ⚠  < 4 GB VRAM — packing 1B BF16 weights needs ~2 GB.  Use T4 or better.')
else:
    print(f'  ✓ {vram_gb:.1f} GB VRAM — model fits comfortably')

# ── Clone axiom ───────────────────────────────────────────────────────────────
if not AXIOM_DIR.is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    REPO_URL, str(AXIOM_DIR)], check=True)
    print(f'✓ axiom cloned  ({BRANCH})')
else:
    r = subprocess.run(['git', '-C', str(AXIOM_DIR), 'pull', '--ff-only'],
                       capture_output=True, text=True)
    print(f'✓ axiom  {r.stdout.strip() or "up to date"}')

sys.path.insert(0, str(AXIOM_DIR))

# ── Persist AXIOM_MASTER_KEY ──────────────────────────────────────────────────
if KEY_FILE.is_file() and not os.environ.get('AXIOM_MASTER_KEY'):
    os.environ['AXIOM_MASTER_KEY'] = KEY_FILE.read_text().strip()
    print('AXIOM_MASTER_KEY restored from session key file')
elif not os.environ.get('AXIOM_MASTER_KEY'):
    key = secrets.token_hex(32)
    os.environ['AXIOM_MASTER_KEY'] = key
    KEY_FILE.write_text(key)
    print(f'AXIOM_MASTER_KEY generated → {KEY_FILE}')
    print('  ⚠ back this up — same key required to verify the .axm later')
else:
    print('AXIOM_MASTER_KEY already set')

# ── Install deps ──────────────────────────────────────────────────────────────
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'datasets', 'huggingface_hub', 'tqdm'], check=True)
print('✓ deps installed')

# ── Build llama.cpp ───────────────────────────────────────────────────────────
if not (LLAMACPP / 'build' / 'bin' / 'llama-cli').is_file():
    print('Building llama.cpp...  (~10 min on T4)')
    t0 = time.time()
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/ggerganov/llama.cpp', str(LLAMACPP)], check=True)
    subprocess.run(
        ['cmake', '-B', 'build', '-DGGML_CUDA=ON',
         f'-DCMAKE_CUDA_ARCHITECTURES={cuda_arch}'],
        cwd=LLAMACPP, check=True, capture_output=True)
    subprocess.run(
        ['cmake', '--build', 'build', '--config', 'Release', '-j4'],
        cwd=LLAMACPP, check=True, capture_output=True)
    print(f'✓ llama.cpp built  ({time.time()-t0:.0f} s)')
else:
    print('✓ llama.cpp already built')

LLAMA_CLI   = LLAMACPP / 'build' / 'bin' / 'llama-cli'
LLAMA_PPL   = LLAMACPP / 'build' / 'bin' / 'llama-perplexity'
LLAMA_QUANT = LLAMACPP / 'build' / 'bin' / 'llama-quantize'
LLAMA_BENCH = LLAMACPP / 'build' / 'bin' / 'llama-bench'
CONVERT     = next(LLAMACPP.glob('convert*.py'), None)
print(f'✓ ready  llama-cli={LLAMA_CLI.exists()}  llama-perplexity={LLAMA_PPL.exists()}')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Download google/gemma-3-1b-it from HuggingFace
#
# Public model — no HF token required.
# Size: ~2.0 GB (BF16 safetensors).  Time: 3–10 min depending on bandwidth.
# ══════════════════════════════════════════════════════════════════════════════
import json, time
from pathlib import Path
from huggingface_hub import snapshot_download

MODEL_ID  = 'google/gemma-3-1b-it'
MODEL_DIR = OUTPUT_DIR / 'gemma3_1b_hf'

if MODEL_DIR.is_dir() and any(MODEL_DIR.glob('*.safetensors')):
    print(f'✓ model already downloaded  →  {MODEL_DIR}')
else:
    print(f'Downloading {MODEL_ID} ...  (~2.0 GB, BF16)')
    t0 = time.time()
    snapshot_download(
        repo_id   = MODEL_ID,
        local_dir = str(MODEL_DIR),
        ignore_patterns = ['*.pt', '*.bin', '*.ot'],  # safetensors only
    )
    elapsed = time.time() - t0
    files = list(MODEL_DIR.glob('*.safetensors'))
    total_gb = sum(f.stat().st_size for f in files) / 1024**3
    print(f'✓ downloaded  {len(files)} shard(s)  {total_gb:.2f} GB  ({elapsed:.0f} s)')

# Quick architecture check
cfg = json.loads((MODEL_DIR / 'config.json').read_text())
print(f'\nArchitecture:')
print(f'  model_type         : {cfg.get("model_type")}')
print(f'  vocab_size         : {cfg.get("vocab_size"):,}')
print(f'  hidden_size        : {cfg.get("hidden_size")}')
print(f'  num_hidden_layers  : {cfg.get("num_hidden_layers")}')
print(f'  num_attention_heads: {cfg.get("num_attention_heads")}')
print(f'  num_kv_heads       : {cfg.get("num_key_value_heads")}')
print(f'  intermediate_size  : {cfg.get("intermediate_size")}')
print(f'  max_pos_embeddings : {cfg.get("max_position_embeddings"):,}')
print(f'  head_dim           : {cfg.get("head_dim", cfg.get("hidden_size", 0) // cfg.get("num_attention_heads", 1))}')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — SRD-4 pack → gemma3_1b_srd4.axm
#
# SRD-4 = W4 base quantization, ~4.5 bpw at pack time.
# Signs every weight shard with HMAC-SHA256 — fingerprint is a public
# commitment to the exact packed weights.
#
# Time: ~15 min on T4  |  ~6 min on A100
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, sys, time
from pathlib import Path

AXM_PATH    = OUTPUT_DIR / 'gemma3_1b_srd4.axm'
PACK_SCRIPT = AXIOM_DIR / 'research' / 'quant' / 'pack_to_axm.py'

if AXM_PATH.exists():
    print(f'✓ .axm already exists  ({AXM_PATH.stat().st_size/1024**3:.2f} GB)  — skipping pack')
else:
    print(f'Packing {MODEL_ID} → SRD-4 .axm ...')
    t0 = time.time()
    r = subprocess.run(
        [sys.executable, str(PACK_SCRIPT),
         '--model',      str(MODEL_DIR),
         '--output',     str(AXM_PATH),
         '--srd4',                         # SRD-4: W4-only, ~4.5 bpw
         '--group-size', '64'],
        capture_output=True, text=True
    )
    elapsed = time.time() - t0
    if r.returncode != 0:
        print('STDERR:', r.stderr[-2000:])
        raise RuntimeError('pack_to_axm.py failed')
    print(f'✓ packed  ({elapsed/60:.1f} min)')

axm_gb = AXM_PATH.stat().st_size / 1024**3
print(f'  .axm size : {axm_gb:.2f} GB')
print(f'  vs BF16   : ~2.0 GB  →  {100*(1 - axm_gb/2.0):.0f}% smaller')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Verify .axm proof ledger + extract → GGUF Q4_K_M
#
# Step 1: axm_cli.py verify — checks every HMAC-SHA256 proof shard
# Step 2: axm_to_gguf.py   — reconstructs FP16 → convert_hf_to_gguf.py → quantize
#
# Expected GGUF size: ~670 MB  (Q4_K_M, Gemma 3 1B)
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, sys, time
from pathlib import Path

AXM_CLI     = AXIOM_DIR / 'axm_cli.py'
EXTRACT_SCR = AXIOM_DIR / 'research' / 'quant' / 'axm_to_gguf.py'
GGUF_SRD    = OUTPUT_DIR / 'gemma3_1b_srd4_q4km.gguf'

# ── Step 1: Verify ────────────────────────────────────────────────────────────
print('Verifying .axm proof chain...')
r = subprocess.run(
    [sys.executable, str(AXM_CLI), 'verify', str(AXM_PATH)],
    capture_output=True, text=True
)
if r.returncode != 0:
    print('VERIFY FAILED:', (r.stdout + r.stderr)[-1000:])
    raise RuntimeError('Proof verification failed')

try:
    vdata  = json.loads(r.stdout)
    fp     = vdata.get('fingerprint', 'n/a')
    proofs = vdata.get('proofs_checked', 'n/a')
except Exception:
    fp, proofs = 'see output', 'see output'
print(f'  ✓ verified  fingerprint={fp}  proofs={proofs}')

# ── Step 2: Extract to GGUF Q4_K_M ───────────────────────────────────────────
if GGUF_SRD.exists():
    print(f'\n✓ SRD GGUF already exists  ({GGUF_SRD.stat().st_size/1024**2:.0f} MB)')
else:
    print('\nExtracting → GGUF Q4_K_M...  (~10-15 min on T4)')
    t0 = time.time()
    r = subprocess.run(
        [sys.executable, str(EXTRACT_SCR),
         '--container', str(AXM_PATH),
         '--gguf-out',  str(GGUF_SRD),
         '--llamacpp',  str(LLAMACPP),
         '--quant',     'Q4_K_M'],
        capture_output=True, text=True
    )
    elapsed = time.time() - t0
    if r.returncode != 0:
        print('STDERR:', r.stderr[-2000:])
        raise RuntimeError('axm_to_gguf.py failed')
    print(f'✓ extracted  ({elapsed/60:.1f} min)')

gguf_mb      = GGUF_SRD.stat().st_size / 1024**2
expected_mb  = 670
delta_pct    = 100 * (gguf_mb - expected_mb) / expected_mb
print(f'  SRD GGUF size : {gguf_mb:.0f} MB  (expected ~{expected_mb} MB  Δ{delta_pct:+.1f} %)')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Generate MET RAM sidecar (Gemma 3 1B architecture)
#
# Expected hydration budgets (Gemma 3 1B, vocab=262144 → large embedding):
#   Embedding (F16, always pinned) : ~580 MB
#   INFORM (early only)            : ~640 MB
#   HARM (all chunks)              : ~760 MB
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, sys
from pathlib import Path

SIDECAR_PATH = OUTPUT_DIR / 'gemma3_1b_met_sidecar.json'
ESTIMATOR    = AXIOM_DIR / 'research' / 'quant' / 'met_ram_estimator.py'

# Read arch from downloaded config.json for accuracy
cfg    = json.loads((MODEL_DIR / 'config.json').read_text())
vocab  = cfg['vocab_size']
hidden = cfg['hidden_size']
layers = cfg['num_hidden_layers']
heads  = cfg['num_attention_heads']
kv     = cfg.get('num_key_value_heads', heads)
inter  = cfg['intermediate_size']

r = subprocess.run([
    sys.executable, str(ESTIMATOR),
    '--vocab-size',        str(vocab),
    '--hidden-size',       str(hidden),
    '--num-layers',        str(layers),
    '--num-heads',         str(heads),
    '--num-kv-heads',      str(kv),
    '--intermediate-size', str(inter),
    '--bpw',               '4.85',
    '--mlp-style',         'geglu',
    '--model-id',          'google/gemma-3-1b-it',
    '--output',            str(SIDECAR_PATH),
], capture_output=True, text=True)

print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr)

# Summary
sd  = json.loads(SIDECAR_PATH.read_text())
hyd = sd['intent_ram_budget']
emb = sd['embedding_slot']['mb']
inform_mb = hyd['INFORM']['total_mb']
harm_mb   = hyd['HARM']['total_mb']
savings_pct = 100 * (1 - inform_mb / harm_mb)

print(f'\n  Embedding (pinned F16) : {emb:.1f} MB')
print(f'  INFORM budget          : {inform_mb:.1f} MB  ({hyd["INFORM"]["ufs_load_ms"]:.0f} ms load)')
print(f'  HARM budget            : {harm_mb:.1f} MB  ({hyd["HARM"]["ufs_load_ms"]:.0f} ms load)')
print(f'  Savings INFORM vs HARM : {savings_pct:.1f} %')
print(f'  vocab_size={vocab:,} — large Gemma vocab means embedding dominates at ~{emb:.0f} MB F16')
print(f'  Sidecar saved → {SIDECAR_PATH}')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — WikiText-2 PPL: SRD Q4_K_M
#
# Same settings as all other Axiom benchmarks:
#   stride 512, context 2048, 100 chunks
#
# Baseline (same setup): Gemma 3 1B Q4_K_M (standard K-quant) = 28.90
# SRD-4 should match or beat this — if it doesn't, SRD dithering is not
# helping at this scale and bpw.
#
# Time: ~8 min on T4
# ══════════════════════════════════════════════════════════════════════════════
import re, subprocess, sys, time
from pathlib import Path

# Download WikiText-2 test set
WT2_PATH = OUTPUT_DIR / 'wikitext2_test.txt'
if not WT2_PATH.exists():
    from datasets import load_dataset
    ds   = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
    text = '\n'.join(ds['text'])
    WT2_PATH.write_text(text)
    print(f'✓ wikitext-2 saved  ({WT2_PATH.stat().st_size/1024:.0f} KB)')
else:
    print(f'✓ wikitext-2 already downloaded')

print('\nRunning PPL: SRD Q4_K_M  (~8 min on T4)...')
t0 = time.time()
r  = subprocess.run([
    str(LLAMA_PPL),
    '-m',             str(GGUF_SRD),
    '-f',             str(WT2_PATH),
    '--ctx-size',     '2048',
    '--ppl-stride',   '512',
    '--chunks',       '100',
    '--n-gpu-layers', '99',
], capture_output=True, text=True)
elapsed_srd = time.time() - t0

ppl_line = [l for l in (r.stdout + r.stderr).splitlines()
            if 'Perplexity' in l or 'ppl' in l.lower()]
print(f'PPL output ({elapsed_srd/60:.1f} min):')
for l in ppl_line:
    print(f'  {l.strip()}')

ppl_srd = None
ppl_match = re.search(r'Perplexity.*?([0-9]+\.[0-9]+)', r.stdout + r.stderr)
if ppl_match:
    ppl_srd = float(ppl_match.group(1))
    print(f'\n  SRD Q4_K_M PPL : {ppl_srd:.2f}')
    print(f'  Baseline Q4_K_M: 28.90  (standard K-quant, same settings)')
    delta = ppl_srd - 28.90
    if delta < 0:
        print(f'  SRD wins by {abs(delta):.2f} PPL vs standard Q4_K_M ✓')
    elif delta < 0.5:
        print(f'  SRD within noise floor of Q4_K_M (Δ={delta:+.2f})')
    else:
        print(f'  Standard Q4_K_M wins by {delta:.2f} PPL — check SRD pack settings')

# Save for Cell 8 comparison
import json
srd_result = {'ppl': ppl_srd, 'elapsed_min': round(elapsed_srd/60, 1)}
(OUTPUT_DIR / '_srd_ppl_tmp.json').write_text(json.dumps(srd_result))


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — Download Google QAT GGUF + run WikiText-2 PPL
#
# Downloads google/gemma-3-1b-it-qat-q4_0-gguf from HuggingFace.
# This is Google's official QAT (Quantization-Aware Training) GGUF —
# weights trained with fake-quant ops at Q4_0 precision.
#
# Q4_0 bpw = 4.0  vs  SRD Q4_K_M bpw = 4.85
# SRD has more bits; QAT has training-time optimization.
#
# Time: ~5 min download + ~8 min PPL on T4
# ══════════════════════════════════════════════════════════════════════════════
import json, re, subprocess, sys, time
from pathlib import Path
from huggingface_hub import hf_hub_download

QAT_REPO_ID = 'google/gemma-3-1b-it-qat-q4_0-gguf'
GGUF_QAT    = OUTPUT_DIR / 'gemma3_1b_qat_q4_0.gguf'

# ── Download QAT GGUF ─────────────────────────────────────────────────────────
if GGUF_QAT.exists():
    print(f'✓ QAT GGUF already downloaded  ({GGUF_QAT.stat().st_size/1024**2:.0f} MB)')
else:
    print(f'Downloading QAT GGUF from {QAT_REPO_ID} ...')
    t0 = time.time()
    # Try to find the GGUF filename in the repo
    try:
        from huggingface_hub import list_repo_files
        gguf_files = [f for f in list_repo_files(QAT_REPO_ID) if f.endswith('.gguf')]
        print(f'  Found GGUF files: {gguf_files}')
        gguf_filename = gguf_files[0] if gguf_files else 'gemma-3-1b-it-q4_0.gguf'
    except Exception:
        gguf_filename = 'gemma-3-1b-it-q4_0.gguf'

    local_path = hf_hub_download(
        repo_id   = QAT_REPO_ID,
        filename  = gguf_filename,
        local_dir = str(OUTPUT_DIR),
    )
    # Rename to our canonical path
    Path(local_path).rename(GGUF_QAT)
    elapsed = time.time() - t0
    print(f'✓ downloaded  {GGUF_QAT.stat().st_size/1024**2:.0f} MB  ({elapsed:.0f} s)')

# ── Run PPL on QAT GGUF ───────────────────────────────────────────────────────
WT2_PATH = OUTPUT_DIR / 'wikitext2_test.txt'
if not WT2_PATH.exists():
    from datasets import load_dataset
    ds   = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
    text = '\n'.join(ds['text'])
    WT2_PATH.write_text(text)

print('\nRunning PPL: QAT Q4_0  (~8 min on T4)...')
t0 = time.time()
r  = subprocess.run([
    str(LLAMA_PPL),
    '-m',             str(GGUF_QAT),
    '-f',             str(WT2_PATH),
    '--ctx-size',     '2048',
    '--ppl-stride',   '512',
    '--chunks',       '100',
    '--n-gpu-layers', '99',
], capture_output=True, text=True)
elapsed_qat = time.time() - t0

ppl_line = [l for l in (r.stdout + r.stderr).splitlines()
            if 'Perplexity' in l or 'ppl' in l.lower()]
print(f'PPL output ({elapsed_qat/60:.1f} min):')
for l in ppl_line:
    print(f'  {l.strip()}')

ppl_qat = None
ppl_match = re.search(r'Perplexity.*?([0-9]+\.[0-9]+)', r.stdout + r.stderr)
if ppl_match:
    ppl_qat = float(ppl_match.group(1))
    print(f'\n  QAT Q4_0 PPL : {ppl_qat:.2f}')

qat_result = {'ppl': ppl_qat, 'elapsed_min': round(elapsed_qat/60, 1)}
(OUTPUT_DIR / '_qat_ppl_tmp.json').write_text(json.dumps(qat_result))


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 — Side-by-side comparison + llama-bench speed + save results
#
# Compares SRD Q4_K_M vs Google QAT Q4_0 on:
#   - WikiText-2 PPL (lower is better)
#   - File size (MB)
#   - bpw
#   - Token generation speed (llama-bench, pp=512, tg=128, 3 reps)
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, sys, time
from pathlib import Path

# ── Load PPL results from Cells 6 + 7 ────────────────────────────────────────
srd_tmp = json.loads((OUTPUT_DIR / '_srd_ppl_tmp.json').read_text())
qat_tmp = json.loads((OUTPUT_DIR / '_qat_ppl_tmp.json').read_text())
ppl_srd = srd_tmp.get('ppl')
ppl_qat = qat_tmp.get('ppl')

srd_mb   = GGUF_SRD.stat().st_size / 1024**2 if GGUF_SRD.exists() else 0
qat_mb   = GGUF_QAT.stat().st_size / 1024**2 if GGUF_QAT.exists() else 0
base_mb  = 815  # standard Q4_K_M Gemma 3 1B from prior benchmark

# ── llama-bench speed (both models) ──────────────────────────────────────────
bench_results = {}
for label, gguf_path in [('SRD Q4_K_M', GGUF_SRD), ('QAT Q4_0', GGUF_QAT)]:
    if not gguf_path.exists() or not LLAMA_BENCH.exists():
        bench_results[label] = {'tg_ts': None, 'pp_ts': None}
        continue
    print(f'Running llama-bench: {label}...')
    r = subprocess.run([
        str(LLAMA_BENCH),
        '-m', str(gguf_path),
        '-p', '512',    # prefill tokens
        '-n', '128',    # generation tokens
        '-r', '3',      # repetitions
        '--n-gpu-layers', '99',
        '--output', 'json',
    ], capture_output=True, text=True)
    try:
        bd = json.loads(r.stdout)
        results_list = bd if isinstance(bd, list) else bd.get('results', [])
        tg = next((x['avg_ts'] for x in results_list if x.get('n_gen', 0) > 0), None)
        pp = next((x['avg_ts'] for x in results_list if x.get('n_prompt', 0) > 0), None)
        bench_results[label] = {'tg_ts': round(tg, 1) if tg else None,
                                  'pp_ts': round(pp, 1) if pp else None}
    except Exception as e:
        # Fallback: print raw output
        print(f'  bench output: {r.stdout[:300]}')
        bench_results[label] = {'tg_ts': None, 'pp_ts': None}

# ── Print comparison table ────────────────────────────────────────────────────
print()
print('═' * 72)
print('  GEMMA 3 1B — SRD Q4_K_M vs Google QAT Q4_0 Comparison')
print('═' * 72)

rows = [
    ('Quantization method',  'SRD-4 (post-training)', 'QAT (trained w/ fake-quant)'),
    ('bpw',                  '~4.85',                 '~4.0'),
    ('Size (MB)',             f'{srd_mb:.0f}' if srd_mb else 'n/a',
                              f'{qat_mb:.0f}' if qat_mb else 'n/a'),
    ('WikiText-2 PPL',        f'{ppl_srd:.2f}' if ppl_srd else 'run Cell 6',
                              f'{ppl_qat:.2f}' if ppl_qat else 'run Cell 7'),
    ('Standard Q4_K_M PPL',  '28.90 (baseline)',      '28.90 (baseline)'),
    ('TG t/s (GPU)',          str(bench_results.get('SRD Q4_K_M', {}).get('tg_ts') or 'n/a'),
                              str(bench_results.get('QAT Q4_0', {}).get('tg_ts') or 'n/a')),
    ('PP t/s (GPU)',          str(bench_results.get('SRD Q4_K_M', {}).get('pp_ts') or 'n/a'),
                              str(bench_results.get('QAT Q4_0', {}).get('pp_ts') or 'n/a')),
    ('Training change?',      'No (drop-in)',           'Yes (re-train with quant)'),
    ('HMAC governance',       'Yes (fingerprinted)',    'No'),
    ('Re-training required',  'No',                    'Yes (fine-tunes need redo)'),
]

print(f'  {"Metric":<28} {"SRD Q4_K_M":<28} {"Google QAT Q4_0":<22}')
print(f'  {"-"*28} {"-"*28} {"-"*22}')
for metric, srd_val, qat_val in rows:
    print(f'  {metric:<28} {srd_val:<28} {qat_val:<22}')

# ── Verdict ───────────────────────────────────────────────────────────────────
print()
if ppl_srd and ppl_qat:
    delta = ppl_srd - ppl_qat
    if delta < -0.5:
        verdict = (f'SRD wins PPL by {abs(delta):.2f} despite higher bpw — dithering '
                   f'outperforms QAT at 1B scale.')
    elif delta > 0.5:
        verdict = (f'QAT wins PPL by {delta:.2f} at 0.85 fewer bpw — training-time '
                   f'optimization beats post-training dithering at 1B scale.')
    else:
        verdict = (f'PPL delta = {delta:+.2f} (noise floor). Methods are equivalent at 1B scale — '
                   f'SRD wins on governance (no retraining, HMAC provenance).')
    print(f'  Verdict: {verdict}')

# ── Save results JSON ─────────────────────────────────────────────────────────
results_data = {
    'model':      'google/gemma-3-1b-it',
    'timestamp':  time.strftime('%Y-%m-%dT%H:%M:%S'),
    'srd_q4km': {
        'quant':   'SRD-4 → Q4_K_M',
        'bpw':     4.85,
        'size_mb': round(srd_mb),
        'ppl':     ppl_srd,
        'tg_ts':   bench_results.get('SRD Q4_K_M', {}).get('tg_ts'),
        'pp_ts':   bench_results.get('SRD Q4_K_M', {}).get('pp_ts'),
    },
    'qat_q4_0': {
        'quant':   'Google QAT Q4_0',
        'bpw':     4.0,
        'size_mb': round(qat_mb),
        'ppl':     ppl_qat,
        'tg_ts':   bench_results.get('QAT Q4_0', {}).get('tg_ts'),
        'pp_ts':   bench_results.get('QAT Q4_0', {}).get('pp_ts'),
    },
    'baseline_q4km_ppl': 28.90,
    'benchmark_config': {
        'ppl_dataset': 'wikitext-2',
        'ppl_stride':  512,
        'ppl_context': 2048,
        'ppl_chunks':  100,
        'speed_pp':    512,
        'speed_tg':    128,
        'speed_reps':  3,
    },
}

RESULTS_PATH = OUTPUT_DIR / 'gemma3_1b_srd_vs_qat.json'
RESULTS_PATH.write_text(json.dumps(results_data, indent=2))
print(f'\n  Results saved → {RESULTS_PATH}')

# ── Download prompt ───────────────────────────────────────────────────────────
print(f'''
  Output files:
    {GGUF_SRD.name:<40}  {srd_mb:.0f} MB  (SRD Q4_K_M)
    {GGUF_QAT.name:<40}  {qat_mb:.0f} MB  (Google QAT Q4_0)
    {SIDECAR_PATH.name:<40}  MET sidecar
    {RESULTS_PATH.name:<40}  benchmark JSON
''')

try:
    from google.colab import files
    for f in [GGUF_SRD, SIDECAR_PATH, RESULTS_PATH]:
        if f.exists():
            files.download(str(f))
except ImportError:
    print(f'  cp {GGUF_SRD} /your/local/path/')
    print(f'  cp {RESULTS_PATH} results/')
